In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pyarrow.dataset as ds
import geopandas as gpd
import matplotlib.pyplot as plt
import folium
from branca.colormap import linear
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ============================================================
# CONFIG
# ============================================================
DATA_DIR       = "stgcn_dataset"
MODEL_PATH     = "best_stgcn_model_tweedie.pth"
SHAPEFILE_PATH = "taxi_zones/geo_export.shp"
OUTPUT_DIR     = "eval_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_NODES  = 263
SEQ_LEN    = 24
PRED_LEN   = 1
BATCH_SIZE = 64
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from train_stgcn_tweedie import FastMultiGraphSTGCN, SpatioTemporalDataset

Using device: cuda


In [2]:
def load_artifacts():
    num_features = int(np.load(os.path.join(DATA_DIR, "num_features.npy")))
    feature_cols = list(np.load(os.path.join(DATA_DIR, "feature_cols.npy"), allow_pickle=True))
    targets      = list(np.load(os.path.join(DATA_DIR, "targets.npy"),      allow_pickle=True))
    time_bins    = np.load(os.path.join(DATA_DIR, "time_bins.npy"),         allow_pickle=True)

    num_timesteps = len(time_bins)
    num_targets   = len(targets)
    total_rows    = num_timesteps * NUM_NODES

    X_mm = np.memmap(os.path.join(DATA_DIR, "X_memmap.dat"),
                     dtype='float32', mode='r', shape=(total_rows, num_features))
    Y_mm = np.memmap(os.path.join(DATA_DIR, "Y_memmap.dat"),
                     dtype='float32', mode='r', shape=(total_rows, num_targets))

    A_s = torch.FloatTensor(
        np.load(os.path.join(DATA_DIR, "adj_spatial.npy"))).to(DEVICE)
    A_f = torch.FloatTensor(
        np.load(os.path.join(DATA_DIR, "adj_flow.npy"))).to(DEVICE)

    return (X_mm, Y_mm, time_bins, num_features, num_targets,
            feature_cols, targets, A_s, A_f)

# METRICS
def compute_all_metrics(y_true, y_pred, target_names, time_bins_test):
    """
    y_true, y_pred : (T, N, K) in original scale
    Returns dict of per-target result dicts.
    """
    results = {}
    T, N, K = y_true.shape
    timestamps = pd.to_datetime(time_bins_test)
    hours = timestamps.hour.values
    dow   = timestamps.dayofweek.values

    for k, name in enumerate(target_names):
        yt = y_true[:, :, k]   # (T, N)
        yp = y_pred[:, :, k]

        mae  = np.mean(np.abs(yt - yp))
        rmse = np.sqrt(np.mean((yt - yp) ** 2))
        bias = np.mean(yp - yt)
        eps  = 1.0
        mask = yt > eps
        mape = np.mean(np.abs((yt[mask] - yp[mask]) / yt[mask])) * 100

        peak_mask  = np.isin(hours, [7, 8, 9, 17, 18, 19])
        wkend_mask = dow >= 5

        peak_mae   = np.mean(np.abs((yt - yp)[peak_mask]))
        offpk_mae  = np.mean(np.abs((yt - yp)[~peak_mask]))
        wkday_mae  = np.mean(np.abs((yt - yp)[~wkend_mask]))
        wkend_mae  = np.mean(np.abs((yt - yp)[wkend_mask]))

        # Top-10 hit rate
        topk_hits = [
            len(set(np.argsort(yt[t])[-10:]) & set(np.argsort(yp[t])[-10:])) / 10
            for t in range(T)
        ]

        # Hour × weekday MAE grid (24 × 7)
        hod_dow = np.full((24, 7), np.nan)
        for h in range(24):
            for d in range(7):
                sel = (hours == h) & (dow == d)
                if sel.sum() > 0:
                    hod_dow[h, d] = np.mean(np.abs((yt - yp)[sel]))

        # Demand bucket MAE
        buckets    = [(0, 5), (5, 20), (20, 50), (50, 200), (200, 1e9)]
        bucket_mae = {}
        for lo, hi in buckets:
            m = (yt >= lo) & (yt < hi)
            if m.sum() > 0:
                lbl = f"{lo}-{'∞' if hi > 1e6 else int(hi)}"
                bucket_mae[lbl] = np.mean(np.abs((yt - yp)[m]))

        print(f"\n{'='*54}")
        print(f"  {name.upper()}")
        print(f"{'='*54}")
        print(f"  MAE          : {mae:.3f}")
        print(f"  RMSE         : {rmse:.3f}")
        print(f"  MAPE         : {mape:.2f}%")
        print(f"  Bias         : {bias:+.3f}  ({'over' if bias > 0 else 'under'}-predict)")
        print(f"  Peak MAE     : {peak_mae:.3f}    Off-peak MAE: {offpk_mae:.3f}")
        print(f"  Weekday MAE  : {wkday_mae:.3f}   Weekend MAE : {wkend_mae:.3f}")
        print(f"  Top-10 Hit   : {np.mean(topk_hits):.3f}")

        results[name] = dict(
            yt=yt, yp=yp,
            per_node_mae =np.mean(np.abs(yt - yp), axis=0),   # (N,)
            per_node_bias=np.mean(yp - yt,         axis=0),
            hod_dow=hod_dow,
            bucket_mae=bucket_mae,
        )

    return results


# PLOTS
DARK = dict(
    bg="#0f1117", panel="#1a1d27", grid="#2a2d3a",
    text="#e8eaf0", sub="#7b8099",
    c1="#4fc3f7", c2="#ff6b6b", c3="#a8edaa",
)

plt.rcParams.update({
    "figure.facecolor": DARK["bg"],  "axes.facecolor":  DARK["panel"],
    "axes.edgecolor":   DARK["grid"],"axes.labelcolor": DARK["text"],
    "xtick.color":      DARK["sub"], "ytick.color":     DARK["sub"],
    "text.color":       DARK["text"],"grid.color":      DARK["grid"],
    "grid.linestyle":   "--",        "grid.alpha":      0.5,
    "font.family":      "monospace",
})


def _save(fig, fname):
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, fname), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved {fname}")


def plot_scatter(res, name):
    yt, yp  = res["yt"].ravel(), res["yp"].ravel()
    idx     = np.random.choice(len(yt), min(60_000, len(yt)), replace=False)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(yt[idx], yp[idx], s=2, alpha=0.2, color=DARK["c1"], rasterized=True)
    lim = max(yt[idx].max(), yp[idx].max()) * 1.05
    ax.plot([0, lim], [0, lim], color=DARK["c2"], lw=1.5, label="Perfect")
    ax.set(xlabel="Actual", ylabel="Predicted",
           title=f"{name} — Predicted vs Actual")
    ax.legend(fontsize=9)
    ax.grid(True)
    _save(fig, f"scatter_{name}.png")


def plot_hod_dow(res, name):
    grid    = res["hod_dow"]
    fig, ax = plt.subplots(figsize=(10, 7))
    im      = ax.imshow(grid, aspect="auto", cmap="YlOrRd",
                        vmin=np.nanmin(grid),
                        vmax=np.nanpercentile(grid, 95))
    ax.set_xticks(range(7))
    ax.set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
    ax.set_yticks(range(0, 24, 2))
    ax.set_yticklabels([f"{h:02d}:00" for h in range(0, 24, 2)])
    ax.set_title(f"{name} — MAE by Hour × Weekday (2025)")
    fig.colorbar(im, ax=ax, shrink=0.8, label="MAE")
    _save(fig, f"hod_dow_{name}.png")


def plot_bucket_bar(res, name):
    bm      = res["bucket_mae"]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars    = ax.bar(bm.keys(), bm.values(), color=DARK["c1"],
                     edgecolor=DARK["bg"], width=0.6)
    for bar, v in zip(bars, bm.values()):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01 * max(bm.values()),
                f"{v:.2f}", ha="center", va="bottom", fontsize=9)
    ax.set(xlabel="True demand bucket", ylabel="MAE",
           title=f"{name} — MAE by Demand Bucket")
    ax.grid(True, axis="y")
    _save(fig, f"bucket_mae_{name}.png")


def plot_worst_nodes(res, name, top_n=20):
    idx     = np.argsort(res["per_node_mae"])[-top_n:][::-1]
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.barh([f"Node {i}" for i in idx], res["per_node_mae"][idx],
            color=DARK["c2"], edgecolor=DARK["bg"], height=0.7)
    ax.set(xlabel="MAE", title=f"{name} — Top {top_n} Worst Nodes")
    ax.invert_yaxis()
    ax.grid(True, axis="x")
    _save(fig, f"worst_nodes_{name}.png")


def plot_timeseries(res, name, node_ids, time_bins_test):
    yt, yp = res["yt"], res["yp"]
    ts     = pd.to_datetime(time_bins_test)
    fig, axes = plt.subplots(len(node_ids), 1,
                             figsize=(18, 3.5 * len(node_ids)), sharex=False)
    if len(node_ids) == 1:
        axes = [axes]
    for ax, nid in zip(axes, node_ids):
        ax.plot(ts, yt[:, nid], color=DARK["c3"], lw=1.2, alpha=0.9,  label="Actual")
        ax.plot(ts, yp[:, nid], color=DARK["c2"], lw=1.0, alpha=0.85,
                linestyle="--", label="Predicted")
        ax.set_title(f"Node {nid}  |  {name}")
        ax.set_ylabel(name)
        ax.legend(fontsize=8, loc="upper right")
        ax.grid(True)
    fname = f"ts_{name}_nodes{'_'.join(map(str, node_ids))}.png"
    _save(fig, fname)


# FOLIUM CHOROPLETH
def folium_map(per_node_values, node_to_loc, gdf,
               metric_name, title, diverging=False):
    data_dict  = {int(node_to_loc[i]): float(per_node_values[i])
                  for i in range(len(per_node_values))}
    val_series = pd.Series(list(data_dict.values()))

    if diverging:
        limit    = max(abs(val_series.quantile(0.05)),
                       abs(val_series.quantile(0.95))) or 1
        colormap = linear.RdYlGn_11.scale(-limit, limit)
    else:
        v_min    = max(0.01, float(val_series.quantile(0.01)))
        v_max    = max(v_min + 0.01, float(val_series.quantile(0.99)))
        colormap = linear.YlOrRd_09.scale(v_min, v_max)
    colormap.caption = f"{title} — {metric_name}"

    def style_fn(feature):
        try:
            loc_id = int(feature.get('id') or
                         feature['properties'].get('LocationID', 0))
            val    = data_dict.get(loc_id)
        except Exception:
            val = None
        if val is None:
            return {'fillColor': '#2a2d3a', 'color': 'gray',
                    'weight': 0.3, 'fillOpacity': 0.3}
        return {'fillColor': colormap(val), 'color': 'black',
                'weight': 0.5, 'fillOpacity': 0.78}

    m = folium.Map(location=[40.73, -73.93], zoom_start=11,
                   tiles='CartoDB DarkMatter')
    colormap.add_to(m)
    folium.GeoJson(gdf, name=title, style_function=style_fn,
                   tooltip=folium.GeoJsonTooltip(
                       fields=['zone', 'borough'],
                       aliases=['Zone', 'Borough'])).add_to(m)

    fname = os.path.join(OUTPUT_DIR, f"map_{title}_{metric_name}.html")
    m.save(fname)
    print(f"  Saved map_{title}_{metric_name}.html")


In [3]:
# 1. Load artifacts
print("[1] Loading artifacts...")
(X_mm, Y_mm, time_bins, num_features, num_targets,
    feature_cols, targets, adj_spatial, adj_flow) = load_artifacts()

years    = pd.to_datetime(time_bins).year
test_idx = np.where(years == 2025)[0]
print(f"    Test timesteps: {len(test_idx)}")

test_ds     = SpatioTemporalDataset(X_mm, Y_mm, time_bins,
                                    test_idx, SEQ_LEN, PRED_LEN)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=0)

# 2. Load model
print("[2] Loading model...")
model = FastMultiGraphSTGCN(num_features=num_features,
                            hidden_dim=64, out_dim=num_targets).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# 3. Inference
print("[3] Running inference...")
all_preds, all_trues = [], []
with torch.no_grad():
    for batch_x, batch_y in tqdm(test_loader, desc="Inference"):
        preds = model(batch_x.to(DEVICE), adj_spatial, adj_flow)
        all_preds.append(preds.cpu().numpy())
        all_trues.append(batch_y.numpy())

T_test    = len(test_ds.valid_indices)
# Predictions are already in original scale — no inverse_transform needed
all_preds = np.concatenate(all_preds, axis=0).reshape(T_test, NUM_NODES, num_targets)
all_trues = np.concatenate(all_trues, axis=0).reshape(T_test, NUM_NODES, num_targets)

time_bins_test = time_bins[test_ds.valid_indices]

# 4. Metrics
print("\n[4] Computing metrics...")
results = compute_all_metrics(all_trues, all_preds, targets, time_bins_test)

# 5. Static plots
print("\n[5] Generating plots...")
mean_demand = all_trues[:, :, 0].mean(axis=0)
for name in targets:
    res = results[name]
    plot_scatter(res, name)
    plot_hod_dow(res, name)
    plot_bucket_bar(res, name)
    plot_worst_nodes(res, name)
    plot_timeseries(res, name,
                    node_ids=[int(np.argmax(mean_demand)),
                                int(np.argmin(mean_demand)),
                                int(np.argmax(res["per_node_mae"]))],
                    time_bins_test=time_bins_test)

# 6. Folium maps
print("\n[6] Generating Folium maps...")
try:
    gdf      = gpd.read_file(SHAPEFILE_PATH)
    id_table = (ds.dataset(os.path.join(DATA_DIR, "node_features_X.parquet"))
                .to_table(columns=['node_index', 'LocationID']))
    id_df    = (id_table.to_pandas()
                .drop_duplicates('node_index')
                .sort_values('node_index'))
    node_to_loc = dict(zip(id_df['node_index'], id_df['LocationID']))

    for k, name in enumerate(targets):
        res = results[name]
        folium_map(all_preds[:, :, k].mean(axis=0), node_to_loc, gdf,
                    "mean_predicted", name)
        folium_map(res["per_node_mae"],  node_to_loc, gdf, "MAE",  name)
        folium_map(res["per_node_bias"], node_to_loc, gdf, "bias", name, diverging=True)
except Exception as e:
    print(f"  Folium maps skipped ({e}). Check SHAPEFILE_PATH.")

print(f"\nAll outputs saved to: {OUTPUT_DIR}/")


[1] Loading artifacts...


C:\Users\zhanq\AppData\Local\Temp\ipykernel_39836\929248183.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=DE

    Test timesteps: 5712
[2] Loading model...
[3] Running inference...


Inference: 100%|██████████| 90/90 [00:19<00:00,  4.72it/s]



[4] Computing metrics...

  DEMAND
  MAE          : 16.763
  RMSE         : 44.411
  MAPE         : 194.13%
  Bias         : -0.509  (under-predict)
  Peak MAE     : 19.213    Off-peak MAE: 15.946
  Weekday MAE  : 16.273   Weekend MAE : 17.987
  Top-10 Hit   : 0.253

  REVENUE_TOTAL
  MAE          : 379.068
  RMSE         : 1728.214
  MAPE         : 136.16%
  Bias         : -297.544  (under-predict)
  Peak MAE     : 449.280    Off-peak MAE: 355.664
  Weekday MAE  : 382.514   Weekend MAE : 370.452
  Top-10 Hit   : 0.217

[5] Generating plots...
  Saved scatter_demand.png
  Saved hod_dow_demand.png
  Saved bucket_mae_demand.png
  Saved worst_nodes_demand.png
  Saved ts_demand_nodes160_102_160.png
  Saved scatter_revenue_total.png
  Saved hod_dow_revenue_total.png
  Saved bucket_mae_revenue_total.png
  Saved worst_nodes_revenue_total.png
  Saved ts_revenue_total_nodes160_102_131.png

[6] Generating Folium maps...
  Folium maps skipped (taxi_zones/geo_export.shp: No such file or directory